# Chat-CE v1 — fine-tune cross-encoder for LongMemEval chat-domain reranking

Base: `cross-encoder/ms-marco-MiniLM-L-12-v2`  
Loss: BinaryCrossEntropyLoss  
Data: hard-negative pairs from LongMemEval-S 100/400 train split (no test contamination)

**Çalıştırma yeri:**
- **Kaggle:** dataset olarak `task2_train_pairs.jsonl` + `task2_val_pairs.jsonl` ekle. GPU=ON (T4 x2 veya P100). Notebook'u Run All.
- **Colab:** sol panelden iki .jsonl'i yükle (`/content/`'e). Runtime > Change runtime type > T4 GPU. Run All.

Eğitim sonunda en iyi ckpt'yi `chat-ce-v1.zip` olarak indir, Mac'te `~/Projects/adaptmem/checkpoints/chat-ce-v1-20260516/`'a aç.

Beklenen süre: T4 ~5-10 dk (3 epoch x 159 step).

In [ ]:
# 1) Setup
import os, sys, subprocess
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'sentence-transformers==5.4.1', 'torch'])
import torch
print('cuda?', torch.cuda.is_available(), 'device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu')

In [ ]:
# 2) Locate data (Kaggle vs Colab)
from pathlib import Path

candidates = [
    Path('/kaggle/input/chat-ce-v1-20260516/task2_train_pairs.jsonl'),
    Path('/kaggle/input/task2-pairs/task2_train_pairs.jsonl'),
    Path('/content/task2_train_pairs.jsonl'),
    Path('./task2_train_pairs.jsonl'),
]
TRAIN = next(p for p in candidates if p.exists())
VAL = TRAIN.parent / 'task2_val_pairs.jsonl'
assert VAL.exists(), f'val file not found next to {TRAIN}'
print('train:', TRAIN)
print('val:  ', VAL)

In [ ]:
# 3) Load examples
import json
from sentence_transformers import InputExample

def load(p):
    out = []
    for line in open(p):
        r = json.loads(line)
        out.append(InputExample(texts=[r['q'], r['doc']], label=float(r['label'])))
    return out

train_ex = load(TRAIN)
val_ex = load(VAL)
print(f'train={len(train_ex)} val={len(val_ex)}')

In [ ]:
# 4) Train
from sentence_transformers import CrossEncoder
from sentence_transformers.cross_encoder.evaluation import CEBinaryClassificationEvaluator
from torch.utils.data import DataLoader

BASE = 'cross-encoder/ms-marco-MiniLM-L-12-v2'
OUT = './chat-ce-v1-20260516'
EPOCHS = 3
BATCH = 32
LR = 2e-5
MAX_LEN = 384

model = CrossEncoder(BASE, num_labels=1, max_length=MAX_LEN)
loader = DataLoader(train_ex, shuffle=True, batch_size=BATCH)
val_pairs = [[ex.texts[0], ex.texts[1]] for ex in val_ex]
val_labels = [int(ex.label) for ex in val_ex]
evaluator = CEBinaryClassificationEvaluator(val_pairs, val_labels, name='val')
warmup = int(0.1 * len(loader) * EPOCHS)
print(f'epochs={EPOCHS} batch={BATCH} lr={LR} steps/epoch={len(loader)} warmup={warmup}')

model.fit(
    train_dataloader=loader,
    evaluator=evaluator,
    epochs=EPOCHS,
    warmup_steps=warmup,
    optimizer_params={'lr': LR},
    output_path=OUT,
    save_best_model=True,
    show_progress_bar=True,
)
print('saved ->', OUT)

In [ ]:
# 5) Quick sanity-rerank on val (R@1 within val pairs as classifier accuracy)
import numpy as np
from sentence_transformers import CrossEncoder as CE
best = CE(OUT, max_length=MAX_LEN)
scores = best.predict(val_pairs, batch_size=64, show_progress_bar=False)
preds = (np.array(scores) > 0.5).astype(int)
labels = np.array(val_labels)
acc = float((preds == labels).mean())
print(f'val acc (>0.5): {acc:.4f}  pos={int(labels.sum())} neg={int((1-labels).sum())}')

In [ ]:
# 6) Pack for download
import shutil
shutil.make_archive('chat-ce-v1', 'zip', OUT)
print('-> chat-ce-v1.zip')
import os; print('size:', os.path.getsize('chat-ce-v1.zip')//1024, 'KB')